# Prospect Augment Agent — Research + Personalized Outreach with LangGraph
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/9-prospect-searx/prospect_searx_workbook.ipynb)

**Agentic pipelines** chain specialist LLMs so each does only what it is best at — one gathers facts, another writes.
By the end of this workshop you will understand how a **researcher node** uses a web search tool to build a fact
summary, how a **copywriter node** turns those facts into a personalized outreach message, and how typed state
flows between nodes in a LangGraph pipeline.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Architecture** — Two-LLM pipeline: researcher → copywriter |
| 2 | **Data Models** — Pydantic models as typed state contracts |
| 3 | **Sample Prospects** — Hardcoded fictional people across industries |
| 4 | **Research Node** — DuckDuckGo web search + LLM fact extraction |
| 5 | **Copywriting Node** — Structured output with confidence score |
| 6 | **LangGraph Pipeline** — StateGraph with `ProspectState` |
| 7 | **Run It** — Process all sample prospects and print results |
| ★ | **Exercises** — Extend with batch processing and creativity tuning |

---

### Prerequisites
- An `OPENAI_API_KEY` (Colab Secrets panel or a local `.env` file)
- Internet access (the researcher node calls DuckDuckGo)

### Key References
> LangGraph documentation: https://langchain-ai.github.io/langgraph/
>
> Yao, S., et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain-openai",
            "langchain-community",
            "langgraph",
            "duckduckgo-search",
            "pydantic",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid. "
    "Local: add to .env  |  Colab: Secrets panel → 'OPENAI_API_KEY'"
)
print("OPENAI_API_KEY present: True (starts with sk-)")

---

## Part 1 — Architecture: Two-LLM Pipeline

---

### Why Two LLMs?

Asking a single LLM to "research this person and write them an outreach message" mixes two conflicting objectives:

- **Research** requires precision: find verifiable facts, do not fabricate
- **Copywriting** requires creativity: be warm, specific, conversational

Splitting the work into a linear pipeline lets each LLM be given a tightly scoped system prompt optimized for its task.
The researcher uses `temperature=0` for factual extraction; the copywriter can use a higher temperature for more natural prose.

### Pipeline Flow

```
prospect_metadata
       │
       ▼
┌──────────────────────────────────────┐
│          research_node               │
│                                      │
│  1. Build query: name + company      │
│  2. DuckDuckGo search → raw snippets │
│  3. LLM extracts 2-3 key facts       │
│                                      │
│  State update: research (str)        │
└──────────────────┬───────────────────┘
                   │
                   ▼
┌──────────────────────────────────────┐
│          copywrite_node              │
│                                      │
│  Reads: prospect + research          │
│  LLM drafts personalized message     │
│  with_structured_output → typed obj  │
│                                      │
│  State update: output (ProspectOutput│
└──────────────────┬───────────────────┘
                   │
                   ▼
      ProspectOutput
  (generated_message, confidence,
   source_summary)
```

### Why DuckDuckGo in This Workbook?

The original `9-prospect-searx` example uses a self-hosted SearXNG instance — a privacy-focused meta-search engine
that requires a running server. That is not compatible with Google Colab.

This workbook uses **DuckDuckGo** as a drop-in replacement: same search-extract pattern, no API key required,
works anywhere. The underlying pipeline is identical; only the search tool differs.

| Tool | Setup | Rate limits | Best for |
|------|-------|-------------|----------|
| DuckDuckGo | None | Aggressive | Development / workshop |
| SearXNG (original) | Self-hosted server | None (self-controlled) | Privacy-first production |
| Tavily | API key | Generous | LLM-optimized production |

In [ ]:
# ── Data models ───────────────────────────────────────────────────────────────
# Three Pydantic models carry typed data through the pipeline.
# ProspectState is the TypedDict that LangGraph manages as its state object.

from typing import TypedDict

from pydantic import BaseModel, Field


class ProspectMetadata(BaseModel):
    """Input: who we are researching."""

    first_name: str
    last_name: str
    company: str
    position: str


class ProspectOutput(BaseModel):
    """Final pipeline output — filled by copywrite_node."""

    generated_message: str = Field(
        description="Personalized outreach message (<=300 chars)"
    )
    confidence: float = Field(
        ge=0.0, le=1.0, description="Confidence that the message is well-targeted (0–1)"
    )
    source_summary: str = Field(
        description="One sentence summarizing the key fact used in the message"
    )


class ProspectState(TypedDict):
    """The shared state object passed through every node."""

    prospect: ProspectMetadata   # set at startup, never changed
    research: str | None         # filled by research_node
    output: ProspectOutput | None  # filled by copywrite_node


# Demonstrate state initialization
example_state: ProspectState = {
    "prospect": ProspectMetadata(
        first_name="Amara",
        last_name="Osei",
        company="Veridian Biotech",
        position="Head of Clinical Partnerships",
    ),
    "research": None,
    "output": None,
}

print("ProspectState schema:")
print(f"  prospect  = {example_state['prospect']}")
print(f"  research  = {example_state['research']}  (filled by research_node)")
print(f"  output    = {example_state['output']}   (filled by copywrite_node)")

---

## Part 2 — Data Models

---

### The State Is the Contract Between Nodes

LangGraph passes one state object through all nodes. Every node reads from it and returns a dict of *only the fields it updates*. LangGraph merges that dict back into state.

```python
# Node pattern — always the same shape:
def my_node(state: ProspectState) -> dict:
    thing = state["some_field"]
    result = do_something(thing)
    return {"updated_field": result}  # ONLY the fields this node owns
```

### What `with_structured_output` Does

The copywriter node uses `llm.with_structured_output(ProspectOutput)`. This tells the LLM to return JSON that matches the Pydantic schema:

```python
# Without structured output:
result = llm.invoke(messages)             # → AIMessage(content="Here is my message...")
text = result.content                      # plain string

# With structured output:
structured_llm = llm.with_structured_output(ProspectOutput)
result = structured_llm.invoke(messages)  # → ProspectOutput(generated_message=..., confidence=...)
text = result.generated_message            # typed field, validated
```

Benefits:
- `confidence` must be 0.0–1.0 — guaranteed by Pydantic
- `source_summary` forces the LLM to articulate what fact it used
- Type-safe access: `result.generated_message` not `result["generated_message"]`

---

## Part 3 — Sample Prospects

---

The original `9-prospect-searx` reads a LinkedIn connections CSV export and processes each row.
In this workshop we use **hardcoded fictional prospects** so you can run the notebook without any CSV file.

All names, companies, and titles below are invented. The pipeline pattern is identical to what you would use with real LinkedIn data — just swap in your own `ProspectMetadata` list.

In [ ]:
# ── Sample prospects ──────────────────────────────────────────────────────────
# Fictional but realistic — diverse industries, seniority levels, and roles.
# The pipeline will search for "{first_name} {last_name} {company}" on DuckDuckGo.
# Because these people don't exist, the researcher will find general company info instead.

SAMPLE_PROSPECTS = [
    ProspectMetadata(
        first_name="Amara",
        last_name="Osei",
        company="Stripe",  # real company for better search results
        position="Head of Clinical Partnerships",
    ),
    ProspectMetadata(
        first_name="Marco",
        last_name="Delacroix",
        company="Shopify",  # real company for better search results
        position="Director of Merchant Operations",
    ),
    ProspectMetadata(
        first_name="Priya",
        last_name="Krishnamurthy",
        company="Databricks",  # real company for better search results
        position="VP of Data Engineering",
    ),
    ProspectMetadata(
        first_name="Lena",
        last_name="Vogt",
        company="Figma",  # real company for better search results
        position="Senior Product Designer",
    ),
]

print(f"Loaded {len(SAMPLE_PROSPECTS)} sample prospects:\n")
for p in SAMPLE_PROSPECTS:
    print(f"  {p.first_name} {p.last_name} — {p.position} at {p.company}")

---

## Part 4 — Research Node: Web Search + Fact Extraction

---

### What the Research Node Does

1. Reads `state["prospect"]` to build a search query: `"{first_name} {last_name} {company}"`
2. Calls `DuckDuckGoSearchResults` to fetch up to 5 web snippets
3. Passes the raw snippets to an LLM with a strict system prompt: extract 2–3 verifiable facts
4. Returns `{"research": extracted_facts_string}` — only the field it owns

### Why Extract Facts Before Writing?

Feeding raw web snippets directly to the copywriter would:
- Waste tokens (snippets contain ads, navigation text, boilerplate)
- Risk hallucination (copywriter might draw on training data rather than the snippets)
- Mix concerns (researching and writing in one prompt reduces quality for both)

The researcher acts as a **fact distillation layer** — the copywriter only sees clean, attributed sentences.

In [ ]:
# ── Research node ─────────────────────────────────────────────────────────────
import time

from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

_search = DuckDuckGoSearchResults(max_results=5)
_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def research_node(state: ProspectState) -> dict:
    """Search the web and extract 2-3 facts about the prospect and their company."""
    p = state["prospect"]
    query = f"{p.first_name} {p.last_name} {p.company} {p.position}"
    print(f"  [research] searching: {query}")

    # Add a small delay to avoid DuckDuckGo rate limiting
    time.sleep(1)
    try:
        raw = _search.run(query)
    except Exception as e:
        # Fallback: search just by company if the full query fails
        print(f"  [research] full query failed ({e}), retrying with company only")
        time.sleep(3)
        raw = _search.run(f"{p.company} company overview")

    resp = _llm.invoke(
        [
            SystemMessage(
                "Extract 2-3 factual, verifiable sentences about this person or their company "
                "from the search results. Be concise. Do not fabricate or infer — "
                "only include what is explicitly stated in the search results. "
                "If the person is not found, focus on recent company news instead."
            ),
            HumanMessage(
                f"Prospect: {p.first_name} {p.last_name}, "
                f"{p.position} at {p.company}\n\n"
                f"Search results:\n{raw[:2000]}"
            ),
        ]
    )

    # Return ONLY the field this node is responsible for
    return {"research": resp.content}


# Test the research node on one prospect before wiring into the graph
test_state: ProspectState = {
    "prospect": SAMPLE_PROSPECTS[0],
    "research": None,
    "output": None,
}

research_update = research_node(test_state)
print()
print("Extracted research:")
print(research_update["research"])

---

## Part 5 — Copywriting Node: Structured Output

---

### Copywriter System Prompt Design

The copywriter prompt must:
- **Constrain length**: "<=300 characters" — specific and measurable
- **Require specificity**: "reference a concrete fact" — prevents generic messages
- **Prohibit pitching**: "do NOT pitch services" — keeps it conversational
- **Request a source summary**: forces the LLM to articulate which fact it used

### The `confidence` Score

The copywriter is asked to self-report a confidence score between 0 and 1. A low score (< 0.6) usually means:
- The researcher could not find relevant facts about this specific person
- The message had to fall back to generic company-level facts
- The research summary was vague or off-topic

You can use this score to filter or flag outputs before sending them.

In [ ]:
# ── Copywriting node ──────────────────────────────────────────────────────────
_copywriter_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

COPYWRITER_SYSTEM = (
    "You are an expert outreach copywriter. Write a friendly, specific 1-sentence message "
    "(<=300 chars) that references a concrete fact about the prospect or their company. "
    "Do NOT pitch any services. Aim to start a genuine conversation. "
    "If the research is about the company rather than the individual, "
    "frame the message around that company context."
)


def copywrite_node(state: ProspectState) -> dict:
    """Draft a personalized outreach message using the research facts."""
    p = state["prospect"]
    research = state["research"] or "No specific research found."
    print(f"  [copywrite] drafting for {p.first_name} {p.last_name}")

    structured_llm = _copywriter_llm.with_structured_output(ProspectOutput)

    result = structured_llm.invoke(
        [
            SystemMessage(COPYWRITER_SYSTEM),
            HumanMessage(
                f"Prospect: {p.first_name} {p.last_name}, "
                f"{p.position} at {p.company}\n\n"
                f"Research facts:\n{research}"
            ),
        ]
    )

    # Return ONLY the field this node is responsible for
    return {"output": result}


# Test the copywrite node using the research we just fetched
state_after_research = {**test_state, **research_update}
copywrite_update = copywrite_node(state_after_research)

out = copywrite_update["output"]
print()
print(f"Generated message ({len(out.generated_message)} chars):")
print(f"  {out.generated_message}")
print(f"Confidence: {out.confidence:.2f}")
print(f"Source:     {out.source_summary}")

---

## Part 6 — LangGraph Pipeline

---

### LangGraph StateGraph API

```python
from langgraph.graph import StateGraph, START, END

graph = StateGraph(ProspectState)       # declare the state schema
graph.add_node("research", fn)          # register a node
graph.add_edge(START, "research")       # define execution order
graph.add_edge("research", "copywrite")
graph.add_edge("copywrite", END)
app = graph.compile()                   # compile into a runnable
result = app.invoke(initial_state)      # run it
```

This pipeline uses only **direct edges** — the flow is always `research → copywrite`.
The graph is linear by design: each stage has a clear responsibility and they never need to branch.

### State Flow Diagram

```
START
  │  state = {prospect: ..., research: None, output: None}
  ▼
research_node
  │  updates: research = "extracted facts string"
  ▼
copywrite_node
  │  updates: output = ProspectOutput(message, confidence, source)
  ▼
END
  │  final state has all three fields populated
```

In [ ]:
# ── Build the StateGraph ──────────────────────────────────────────────────────
from langgraph.graph import END, START, StateGraph

graph = StateGraph(ProspectState)

# Register nodes
graph.add_node("research", research_node)
graph.add_node("copywrite", copywrite_node)

# Define execution order: START → research → copywrite → END
graph.add_edge(START, "research")
graph.add_edge("research", "copywrite")
graph.add_edge("copywrite", END)

# Compile into a runnable
app = graph.compile()

print("Graph compiled successfully.")
print("Execution order: START → research → copywrite → END")
print()
print("To run the pipeline:")
print('  result = app.invoke({"prospect": p, "research": None, "output": None})')
print('  print(result["output"].generated_message)')

---

## Part 7 — Process All Prospects

---

### Running the Pipeline on Each Prospect

The loop below processes each prospect in `SAMPLE_PROSPECTS`. A small `time.sleep()` between calls avoids
DuckDuckGo rate-limiting.

After each run, we print the research summary, the generated message, and the confidence score.
Results are collected in `all_results` for the batch summary table at the end.

In [ ]:
# ── Run the pipeline on each sample prospect ──────────────────────────────────
all_results = []

for i, prospect in enumerate(SAMPLE_PROSPECTS):
    print(f"\n[{i+1}/{len(SAMPLE_PROSPECTS)}] {prospect.first_name} {prospect.last_name} — "
          f"{prospect.position} at {prospect.company}")
    print("-" * 70)

    try:
        result = app.invoke({
            "prospect": prospect,
            "research": None,
            "output": None,
        })

        out = result["output"]
        research = result["research"]

        print(f"  Research: {research[:120]}..." if len(research) > 120 else f"  Research: {research}")
        print(f"  Message:  {out.generated_message}")
        print(f"  Length:   {len(out.generated_message)} chars")
        print(f"  Confidence: {out.confidence:.2f}")
        print(f"  Source:   {out.source_summary}")

        all_results.append({
            "name": f"{prospect.first_name} {prospect.last_name}",
            "company": prospect.company,
            "position": prospect.position,
            "message": out.generated_message,
            "confidence": out.confidence,
            "chars": len(out.generated_message),
            "source": out.source_summary,
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        all_results.append({
            "name": f"{prospect.first_name} {prospect.last_name}",
            "company": prospect.company,
            "position": prospect.position,
            "message": f"ERROR: {e}",
            "confidence": 0.0,
            "chars": 0,
            "source": "",
        })

    # Avoid DuckDuckGo rate limiting between prospects
    if i < len(SAMPLE_PROSPECTS) - 1:
        time.sleep(2)

# ── Summary table ─────────────────────────────────────────────────────────────
print()
print("=" * 80)
print("BATCH SUMMARY")
print("=" * 80)
print(f"{'Name':<22} {'Company':<12} {'Conf':>5} {'Chars':>5}  Message preview")
print("-" * 80)

CONFIDENCE_THRESHOLD = 0.6
for r in all_results:
    conf = r["confidence"]
    flag = "" if conf >= CONFIDENCE_THRESHOLD else " [LOW]"
    preview = r["message"][:38] + "..." if len(r["message"]) > 38 else r["message"]
    print(f"{r['name']:<22} {r['company']:<12} {conf:>5.2f} {r['chars']:>5}  {preview}{flag}")

---

## Exercises

---

Work through these in order. Each builds on the previous.

**Exercise 1 — Add Your Own Prospect**
Replace the TODO fields below and run the pipeline on someone from your own network.
Evaluate: does the message reference something specific? Would you respond to it?

**Exercise 2 — Process a Larger Batch**
Extend `SAMPLE_PROSPECTS` to 8 prospects and process all of them in the loop.
Add a confidence filter: skip prospects where `confidence < 0.6` and print a SKIPPED notice instead.

**Exercise 3 — Add a LinkedIn Scraper Node**
Add a third node `linkedin_node` between research and copywrite that constructs a hypothetical LinkedIn
profile URL: `https://linkedin.com/in/{first_name}-{last_name}`. Store it in state and include it in the output.
Hint: add `linkedin_url: str | None` to `ProspectState` and `ProspectOutput`.

**Exercise 4 — Vary Copywriter Temperature**
Create two copywriter LLMs: one with `temperature=0` (precise) and one with `temperature=0.9` (creative).
Run both on the same prospect and compare the message quality and tone.
Which temperature produces more compelling outreach?

In [ ]:
# ── Exercise scaffolds ────────────────────────────────────────────────────────

# Exercise 1: Add your own prospect
MY_PROSPECT = ProspectMetadata(
    first_name="TODO",
    last_name="TODO",
    company="TODO",
    position="TODO",
)
# TODO: uncomment and run after filling in MY_PROSPECT
# result = app.invoke({"prospect": MY_PROSPECT, "research": None, "output": None})
# print("Research:", result["research"])
# print("Message:", result["output"].generated_message)
# print("Confidence:", result["output"].confidence)


# Exercise 2: Batch processing with confidence filter
def run_batch_with_filter(prospects, confidence_threshold=0.6):
    """Run the pipeline over a list of prospects and skip low-confidence outputs."""
    results = []
    for i, p in enumerate(prospects):
        print(f"[{i+1}/{len(prospects)}] {p.first_name} {p.last_name}")
        # TODO: invoke the pipeline
        # TODO: check confidence against threshold
        # TODO: append to results or print SKIPPED
        if i < len(prospects) - 1:
            time.sleep(2)
    return results


# Exercise 3: LinkedIn URL node
def linkedin_node(state: ProspectState) -> dict:
    """Construct a hypothetical LinkedIn profile URL for the prospect."""
    p = state["prospect"]
    # TODO: build the URL: https://linkedin.com/in/{first_name}-{last_name} (lowercase)
    # TODO: add 'linkedin_url' to ProspectState and return {"linkedin_url": url}
    return {}


# Exercise 4: Temperature comparison
def compare_temperatures(prospect, research_text):
    """Compare copywriter output at temperature=0 vs temperature=0.9."""
    for temp in [0.0, 0.9]:
        llm_temp = ChatOpenAI(model="gpt-4o-mini", temperature=temp)
        structured = llm_temp.with_structured_output(ProspectOutput)
        # TODO: invoke structured with COPYWRITER_SYSTEM and print the message
        print(f"  temp={temp}: TODO")


print("Exercise scaffolds ready.")
print("Edit and uncomment the TODO sections above to work through each exercise.")

---

*Workshop complete. You built a two-node research → copywriting pipeline with LangGraph, typed state, and
structured LLM output. The same pattern scales to real LinkedIn exports, Brave/Tavily search backends,
and multi-node QA gates — see `examples/3-prospect-agent/` for the full production version.*